#  Curso: Debugging de sistemas agénticos (con Langfuse)

La Clase 7 fue la **vista de arriba** (monitoreo en agregado). Hoy es la **vista de lupa**:
ya sabes que *algo* falló y entras a *una* ejecución a ver **por qué**.

Lo que hace difícil (e interesante) el debugging agéntico son los **ciclos**: un grafo que
puede repetir nodos. Una traza lineal se lee de arriba abajo; un ciclo hay que *desenrollarlo*
para saber qué pasó en cada vuelta. Por eso hoy, por fin, construimos tu **grafo LangGraph con
retry loop** y `search_history`.

> **Monitoreo (Clase 7)** dispara la alerta → **Debugging (hoy)** encuentra la causa raíz.

### Objetivos
- ✅ Construir un grafo con **retry loop**: `retrieve → evaluar → (reformular → reintentar) → generar`
- ✅ Acumular los intentos en `search_history` dentro del estado
- ✅ Instrumentarlo con Langfuse (y conocer el `CallbackHandler` de una línea)
- ✅ Provocar un fallo a propósito: un **loop que no converge**
- ✅ Leer trazas **con ciclos** (timeline y *agent graph view*: agregado vs expandido)
- ✅ **Replay** de una ejecución y **casos de estudio** de fallos típicos


## 1. Instalación

In [ ]:
!pip install transformers torch huggingface_hub rank_bm25 -q
!pip install langgraph langfuse -q

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 2. LLM, sistema base y conexión a Langfuse

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import time
import numpy as np
import torch
from typing import TypedDict, List
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# (si no hiciste login de HuggingFace en esta sesion, descomenta:)
# from huggingface_hub import notebook_login; notebook_login()

print("🔄 Loading LLM...")
model_name = "google/gemma-3-1b-it"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
llm = pipeline("text-generation", model=model, tokenizer=tokenizer)

def ask_llm(prompt, max_new_tokens=60):
    out = llm([{"role": "user", "content": prompt}], max_new_tokens=max_new_tokens,
              do_sample=False, pad_token_id=tokenizer.eos_token_id, return_full_text=False)
    return out[0]["generated_text"].strip()

def count_tokens(text):
    return len(tokenizer.encode(text))

CORPUS = [
    {"id": "doc-01", "text": "CloudBox ofrece 15 GB de almacenamiento gratuito en el plan Free."},
    {"id": "doc-02", "text": "El plan Pro de CloudBox cuesta 9 dolares al mes e incluye 2 TB."},
    {"id": "doc-03", "text": "Para restaurar un archivo borrado, ve a la Papelera; se guardan 30 dias."},
    {"id": "doc-04", "text": "CloudBox cifra los archivos en reposo con AES-256 y en transito con TLS."},
    {"id": "doc-05", "text": "Puedes compartir una carpeta con un enlace de solo lectura o edicion."},
    {"id": "doc-06", "text": "El limite por archivo es 50 GB en Pro y 5 GB en Free."},
    {"id": "doc-07", "text": "CloudBox sincroniza en Windows, macOS, Android e iOS."},
    {"id": "doc-08", "text": "El soporte 24/7 por chat es solo para clientes del plan Pro."},
]
_bm25 = BM25Okapi([d["text"].lower().split() for d in CORPUS])

def retrieve_docs(query, k=3):
    scores = _bm25.get_scores(query.lower().split())
    return [CORPUS[i] for i in np.argsort(scores)[::-1][:k]]

import os
os.environ["LANGFUSE_HOST"]       = "https://us.cloud.langfuse.com"
os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-8c53a053-7a99-4afe-9d78-ffc5ecc10432"
os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-ef93aaf8-901b-4c6b-83e2-ce51c55d9f1c"
# 🔒 rota la secret key cuando termines de probar

from langfuse import get_client, propagate_attributes
langfuse = get_client()
print("✅ Conectado" if langfuse.auth_check() else "❌ auth_check fallo")

🔄 Loading LLM...


config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.00GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

✅ Conectado


## 3. El evaluador de contexto y el reformulador

El retry loop necesita dos piezas nuevas:

- **`grade_context`** — decide si el contexto recuperado es "suficientemente bueno". Usamos
  solapamiento de palabras de contenido (determinista y fácil de razonar; en producción aquí
  iría un LLM-as-judge de la Clase 4).
- **`reformulate`** — reescribe la query para reintentar. En producción es un LLM; aquí una
  versión **determinista** para que la demo sea reproducible (verás siempre los mismos ciclos).

In [ ]:
import re

SPANISH_STOP = {"que","cual","cuanto","como","donde","el","la","los","las","un","una","de",
                "del","por","para","me","mi","lo","es","son","y","o","a","al","en","se","su","hay"}

def _tokens(text):
    """Tokeniza quitando puntuacion (evita falsos matches por comas/acentos)."""
    return re.findall(r"[a-záéíóúñ0-9]+", text.lower())

def content_words(text):
    return [w for w in _tokens(text) if w not in SPANISH_STOP and len(w) > 2]

def grade_context(query, docs, min_overlap=2):
    """True si al menos `min_overlap` palabras de contenido de la query aparecen (exactas) en los docs."""
    corpus_words = set()
    for d in docs:
        corpus_words.update(_tokens(d["text"]))
    q = content_words(query)
    overlap = sum(1 for w in q if w in corpus_words)   # match EXACTO, no substring
    return overlap >= min_overlap, overlap

# En produccion: reformulate() llama a un LLM. Aqui: determinista para demo reproducible.
DEMO_REWRITES = {
    "¿donde veo lo que borre?": "restaurar archivo borrado en la papelera",
}
def reformulate(original_query):
    if original_query in DEMO_REWRITES:
        return DEMO_REWRITES[original_query]      # este caso SI mejora -> loop que converge
    return " ".join(content_words(original_query)) # limpia stopwords; no inventa contexto

print("✅ Evaluador y reformulador listos")

✅ Evaluador y reformulador listos


## 4. El grafo LangGraph con retry loop

El estado (`GraphState`) lleva la query actual, la original, el contador de `attempt`, y el
**`search_history`** que acumula cada intento. El grafo:

```
        START
          │
      ┌─►retrieve──► evaluar ──good?──► generate ──► END
      │                       │no
      └──── reformulate ◄──────┘   (hasta MAX_ATTEMPTS; luego se rinde y genera igual)
```

El ciclo `retrieve ↔ reformulate` es lo que hará interesante la traza. Instrumentamos **cada
nodo** con un span de Langfuse (nombres consistentes → el *agent graph view* agrupará las
repeticiones).

In [ ]:
from langgraph.graph import StateGraph, START, END

MAX_ATTEMPTS = 3
PRICE_IN, PRICE_OUT = 0.0005, 0.0015

class GraphState(TypedDict):
    query: str
    original_query: str
    attempt: int
    docs: list
    good: bool
    overlap: int
    gave_up: bool
    search_history: list
    answer: str

def retrieve_node(state: GraphState):
    with langfuse.start_as_current_observation(as_type="span", name="retrieve") as s:
        docs = retrieve_docs(state["query"])
        good, overlap = grade_context(state["query"], docs)
        entry = {"attempt": state["attempt"], "query": state["query"],
                 "overlap": overlap, "good": good, "doc_ids": [d["id"] for d in docs]}
        s.update(input={"query": state["query"]},
                 output={"good": good, "overlap": overlap, "doc_ids": entry["doc_ids"]},
                 metadata={"attempt": state["attempt"]})
        return {"docs": docs, "good": good, "overlap": overlap,
                "search_history": state["search_history"] + [entry]}

def reformulate_node(state: GraphState):
    with langfuse.start_as_current_observation(as_type="span", name="reformulate") as s:
        new_q = reformulate(state["original_query"])
        s.update(input={"query": state["query"]}, output={"reformulated": new_q},
                 metadata={"attempt": state["attempt"]})
        return {"query": new_q, "attempt": state["attempt"] + 1}

def generate_node(state: GraphState):
    with langfuse.start_as_current_observation(as_type="span", name="generate") as s:
        gave_up = not state["good"]
        contexto = "\n".join(f"- {d['text']}" for d in state["docs"])
        prompt = ("Responde usando SOLO el contexto. Si no esta, di que no lo sabes.\n\n"
                  f"Contexto:\n{contexto}\n\nPregunta: {state['original_query']}\nRespuesta:")
        with langfuse.start_as_current_observation(
                as_type="generation", name="llm", model="gemma-3-1b-it") as gen:
            # se actualiza/guarda en langfuse que prompt se uso
            gen.update(input=prompt)
            # usa la llm local yel prompt para generar respuesta
            answer = ask_llm(prompt)
            n_in, n_out = count_tokens(prompt), count_tokens(answer)
            cost = n_in/1000*PRICE_IN + n_out/1000*PRICE_OUT
            gen.update(output=answer,
                       usage_details={"input": n_in, "output": n_out, "total": n_in+n_out},
                       cost_details={"total": cost})
        s.update(output={"answer": answer}, metadata={"gave_up": gave_up})
        return {"answer": answer, "gave_up": gave_up}

def route_after_retrieve(state: GraphState):
    if state["good"]:
        return "generate"                       # contexto bueno -> a generar
    if state["attempt"] >= MAX_ATTEMPTS:
        return "generate"                       # se rinde: genera igual (el bug a observar)
    return "reformulate"                        # reintenta

g = StateGraph(GraphState)
g.add_node("retrieve", retrieve_node)
g.add_node("reformulate", reformulate_node)
g.add_node("generate", generate_node)
g.add_edge(START, "retrieve")
g.add_conditional_edges("retrieve", route_after_retrieve,
                        {"generate": "generate", "reformulate": "reformulate"})
g.add_edge("reformulate", "retrieve")           # 👈 el ciclo
g.add_edge("generate", END)
app = g.compile()
print("✅ Grafo compilado (con ciclo retrieve <-> reformulate)")

✅ Grafo compilado (con ciclo retrieve <-> reformulate)


## 5. Envolver la ejecución en una traza

Cada nodo ya abre su span; aquí envolvemos todo el `invoke` en un span raíz con
`propagate_attributes` (para `session_id`/`user_id`). Los spans de los nodos anidan solos
bajo el raíz por el contexto de OpenTelemetry.

In [ ]:
def run_graph(q, session_id="clase8-demo", user_id="user-0"):
    init = {"query": q, "original_query": q, "attempt": 0, "docs": [], "good": False,
            "overlap": 0, "gave_up": False, "search_history": [], "answer": ""}
    with langfuse.start_as_current_observation(as_type="span", name="cloudbox_graph") as root:
        root.update(input={"query": q})
        with propagate_attributes(session_id=session_id, user_id=user_id,
                                  trace_name="cloudbox_graph"):
            final = app.invoke(init)
        root.update(output={"answer": final["answer"], "attempts": final["attempt"],
                            "gave_up": final["gave_up"]})
    return final

def show_history(final):
    print(f"  intentos: {final['attempt']}   se rindio: {final['gave_up']}")
    for h in final["search_history"]:
        flag = "✓ good" if h["good"] else "✗ bad "
        print(f"   #{h['attempt']} [{flag}] overlap={h['overlap']}  q='{h['query']}'  {h['doc_ids']}")
    print(f"   respuesta: {final['answer'][:90]}")

print("✅ Runner listo")

✅ Runner listo


---
## PARTE 1 — Tres ejecuciones: directa, loop que converge, loop que falla

Corremos tres queries elegidas para que veas los tres comportamientos (deterministas):

1. **Directa** — contexto bueno al primer intento, sin ciclo.
2. **Loop que converge** — mal contexto, reformula **una** vez y se recupera.
3. **Loop que NO converge** — fuera de dominio; reformula hasta `MAX_ATTEMPTS` y **se rinde**.

In [ ]:
casos = {
    "DIRECTA (sin loop)":       "¿Cuanto cuesta el plan Pro?",
    "LOOP QUE CONVERGE":        "¿donde veo lo que borre?",
    "LOOP QUE NO CONVERGE 🐛":  "¿Cual es la capital de Francia?",
}

finales = {}
for etiqueta, q in casos.items():
    print(f"\n===== {etiqueta} =====")
    final = run_graph(q, user_id="user-demo")
    finales[etiqueta] = final
    show_history(final)

langfuse.flush()
print("\n✅ 3 trazas enviadas a Langfuse. Compáralas en Tracing.")

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'pad_token_id', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



===== DIRECTA (sin loop) =====


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GemmaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  intentos: 0   se rindio: False
   #0 [✓ good] overlap=3  q='¿Cuanto cuesta el plan Pro?'  ['doc-02', 'doc-01', 'doc-08']
   respuesta: 9 dolares al mes.

===== LOOP QUE CONVERGE =====


[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  intentos: 1   se rindio: False
   #0 [✗ bad ] overlap=0  q='¿donde veo lo que borre?'  ['doc-08', 'doc-07', 'doc-06']
   #1 [✓ good] overlap=4  q='restaurar archivo borrado en la papelera'  ['doc-03', 'doc-06', 'doc-07']
   respuesta: Papelera

===== LOOP QUE NO CONVERGE 🐛 =====
  intentos: 3   se rindio: True
   #0 [✗ bad ] overlap=0  q='¿Cual es la capital de Francia?'  ['doc-03', 'doc-08', 'doc-06']
   #1 [✗ bad ] overlap=0  q='capital francia'  ['doc-08', 'doc-07', 'doc-06']
   #2 [✗ bad ] overlap=0  q='capital francia'  ['doc-08', 'doc-07', 'doc-06']
   #3 [✗ bad ] overlap=0  q='capital francia'  ['doc-08', 'doc-07', 'doc-06']
   respuesta: No lo sé.

✅ 3 trazas enviadas a Langfuse. Compáralas en Tracing.


---
## PARTE 2 — Leer trazas con ciclos en Langfuse

Abre las tres trazas en `Tracing`. Lo que verás:

- **DIRECTA** — árbol corto: `cloudbox_graph → retrieve → generate`. Un solo `retrieve`.
- **CONVERGE** — `retrieve → reformulate → retrieve → generate`. **Dos** `retrieve` y un
  `reformulate`: el ciclo dio una vuelta y se recuperó.
- **NO CONVERGE** — `retrieve → reformulate` repetidos `MAX_ATTEMPTS` veces y luego `generate`
  con `gave_up=True`. El árbol lineal se hace largo; aquí es donde entra el *agent graph view*.

### El *agent graph view*
En la traza, cambia a la vista de grafo:
- **Aggregated** — dibuja el bucle como un **ciclo** y cuenta las repeticiones
  (`retrieve ×4`). Ideal para ver *de un vistazo* que algo gira de más.
- **Expanded** — **desenrolla** cada vuelta en orden, para inspeccionar intento por intento.

La señal de bug: en la traza NO-CONVERGE, `retrieve` se repite el máximo de veces y `generate`
sale con `gave_up=True`. El `overlap` en cada intento (que registramos como metadata) te dice
*por qué* nunca pasó el evaluador: el contexto jamás matcheó la pregunta.

---
## PARTE 3 — Replay: reproducir el fallo

"Replay" = volver a ejecutar exactamente la misma entrada para reproducir el bug de forma
fiable. Como el grafo es determinista, la traza se repite: mismo número de vueltas, mismo
desenlace. Eso es lo que te permite arreglar con confianza y verificar la corrección.

In [ ]:
print("Replay de la ejecucion que fallo:\n")
replay = run_graph("¿Cual es la capital de Francia?", user_id="user-demo", session_id="clase8-replay")
langfuse.flush()
show_history(replay)

print("\n¿Reproduce el bug?",
      "SI — mismo nº de intentos y gave_up=True" if replay["gave_up"] else "no")
print("En la UI de Langfuse tambien puedes reenviar el prompt de 'llm' al Playground",
      "para probar variantes sin tocar el codigo.")

[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Replay de la ejecucion que fallo:

  intentos: 3   se rindio: True
   #0 [✗ bad ] overlap=0  q='¿Cual es la capital de Francia?'  ['doc-03', 'doc-08', 'doc-06']
   #1 [✗ bad ] overlap=0  q='capital francia'  ['doc-08', 'doc-07', 'doc-06']
   #2 [✗ bad ] overlap=0  q='capital francia'  ['doc-08', 'doc-07', 'doc-06']
   #3 [✗ bad ] overlap=0  q='capital francia'  ['doc-08', 'doc-07', 'doc-06']
   respuesta: No lo sé.

¿Reproduce el bug? SI — mismo nº de intentos y gave_up=True
En la UI de Langfuse tambien puedes reenviar el prompt de 'llm' al Playground para probar variantes sin tocar el codigo.


---
## PARTE 4 — Casos de estudio: patrones de fallo y cómo la traza los delata

| Patrón de fallo | Cómo se ve en la traza | Dónde mirar |
|---|---|---|
| **Loop que no converge** | `retrieve/reformulate` repetidos hasta el máximo; `gave_up=True` | el `overlap` por intento nunca sube → el evaluador o la reformulación fallan |
| **Contexto vacío** | `retrieve` devuelve `doc_ids=[]` o irrelevantes, pero `good=True` | evaluador demasiado laxo (`min_overlap` bajo) → ajusta el umbral |
| **Rama equivocada** | `generate` se ejecuta con `good=False` sin agotar intentos | bug en `route_after_retrieve` (condición mal escrita) |
| **Coste disparado** | muchas vueltas × una `generation` cara por vuelta | el loop es el culpable del coste, visible en el agregado (Clase 7) |

La moraleja del bloque: **el monitoreo (Clase 7) te trae aquí** — una alerta de coste o de
tasa de error apunta a un puñado de trazas — y **el debugging (hoy) las abre** para encontrar,
en el árbol y en `search_history`, la vuelta exacta donde se rompió.

---
## PARTE 5 — (Alternativa) El `CallbackHandler` de una línea

Arriba instrumentamos **a mano** cada nodo (control total, tokens/costo, anidado fiable). La
forma *estándar de producción* para LangChain/LangGraph es el `CallbackHandler` de Langfuse:
captura **cada nodo del grafo automáticamente** sin tocar el código de los nodos.

Si usas el callback, puedes quitar los spans manuales de los nodos; conserva solo el span
`generation` dentro de `generate` si quieres tokens/costo explícitos.

In [ ]:
from langchain_core.runnables import RunnableLambda
from langfuse.langchain import CallbackHandler

# Dos pasos encadenados: normalizar la pregunta -> "responder"
def normalizar(q: str) -> str:
    return q.strip().lower()

def responder(q: str) -> str:
    return f"Recibi la pregunta: '{q}'"

cadena = RunnableLambda(normalizar) | RunnableLambda(responder)

# Sin tocar el codigo de arriba, el callback traza toda la cadena:
handler = CallbackHandler()
with propagate_attributes(trace_name="demo_callback", session_id="ejemplo", user_id="ana"):
    resultado = cadena.invoke("  ¿CUANTO cuesta el plan Pro?  ",
                              config={"callbacks": [handler]})

langfuse.flush()
print(resultado)

Recibi la pregunta: '¿cuanto cuesta el plan pro?'


In [ ]:
# --- Forma estandar de produccion (una linea instrumenta todo el grafo) ---
from langfuse.langchain import CallbackHandler

def run_graph_callback(q, session_id="clase8-cb", user_id="user-0"):
    init = {"query": q, "original_query": q, "attempt": 0, "docs": [], "good": False,
            "overlap": 0, "gave_up": False, "search_history": [], "answer": ""}
    with propagate_attributes(session_id=session_id, user_id=user_id, trace_name="cloudbox_graph_cb"):
        handler = CallbackHandler()                       # 👈 el callback
        final = app.invoke(init, config={"callbacks": [handler]})   # 👈 una linea
    return final

demo = run_graph_callback("¿Cual es la capital de Francia?")
langfuse.flush()
print("✅ Grafo trazado via CallbackHandler. Compara este trace con el manual en la UI.")
print("   (Nota: aqui los nodos se instrumentan DOBLE porque tus nodos ya abren spans;")
print("    en un proyecto real usarias UNA de las dos vias, no ambas.)")

[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ Grafo trazado via CallbackHandler. Compara este trace con el manual en la UI.
   (Nota: aqui los nodos se instrumentan DOBLE porque tus nodos ya abren spans;
    en un proyecto real usarias UNA de las dos vias, no ambas.)


---

### Ejercicios
1. **Rompe la rama.** Cambia `route_after_retrieve` para que use `>` en vez de `>=` en el
   límite de intentos. Corre la query fuera de dominio y encuentra el bug leyendo la traza.
2. **Evaluador laxo.** Baja `min_overlap` a 1 y observa cómo desaparece el loop pero aumentan
   las respuestas malas (`good=True` con contexto flojo). Es el trade-off del grader.
3. **Reformulación real.** Sustituye `reformulate` por una llamada al LLM que reescriba la
   query, e instrumenta esa llamada como una `generation` aparte. ¿Cuánto sube el costo por
   vuelta?
4. **Del monitoreo al debugging.** Reusa la alerta de la Clase 7: cuando `tasa_posible_drift`
   supere el umbral, recoge los `trace_id` afectados y ábrelos. Ese es el flujo real de un
   on-call de LLMOps.
